# narrative_dna Colab Quickstart

Este notebook descarga el repo desde GitHub, instala `narrative_dna`, importa los módulos principales y ejecuta un flujo JSON-first conservador. La notación compacta se deriva siempre desde JSON validado.

## 1. Clonar el repo desde GitHub

In [1]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/jcval94/ADNarrativa.git"
REPO_DIR = Path("/content/ADNarrativa")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print(Path.cwd())

/content/ADNarrativa


## 2. Instalar el paquete

In [2]:
%pip install -q -e ".[dev]"

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for narrative-dna (pyproject.toml) ... done


## 3. Importar módulos principales

In [5]:
import json
from pathlib import Path
import sys
import os

REPO_DIR = Path("/content/ADNarrativa") # Define REPO_DIR again for this cell's context
sys.path.insert(0, str(REPO_DIR / "src")) # Add the src directory to sys.path

from narrative_dna.evaluator import load_gold_units
from narrative_dna.loader import load_documents
from narrative_dna.pipeline import run_pipeline

print("Imports OK")

Imports OK


## 4. Cargar una transcripción de ejemplo sin LLM

In [6]:
documents = load_documents("data/transcripts/videos", limit=1)
document = documents[0]

print("document_id:", document.document_id)
print("units:", len(document.units))
print(document.units[0].model_dump(mode="json"))

document_id: -1wOnr7JsS4
units: 33
{'document_id': '-1wOnr7JsS4', 'unit_id': '-1wOnr7JsS4_u00000', 'sequence_index': 0, 'text': 'Los aviones tienen algo que nosotros no tenemos.', 'normalized_text': 'los aviones tienen algo que nosotros no tenemos.', 'start_ms': 0, 'end_ms': 2400, 'char_start': None, 'char_end': None, 'previous_unit_id': None, 'next_unit_id': '-1wOnr7JsS4_u00001', 'functions': ['N'], 'primary_function': 'N', 'secondary_functions': [], 'inherited_functions': [], 'certainty': 'none', 'emotion_expressed': 'N', 'emotion_intensity': 0, 'emotions_mentioned': [], 'stance': 'neutral', 'target': None, 'speech_act': None, 'logic': None, 'evidence_spans': [], 'rejected_labels': [], 'validator_flags': [], 'heuristic_candidates': [], 'llm_votes': [], 'confidence': 0.0, 'method': 'none', 'needs_review': False, 'review_reasons': [], 'review_status': 'pending', 'final_notation': 'N_N0{0}', 'taxonomy_version': 'v1_0', 'prompt_version': 'v1_0', 'validator_version': 'v1_0'}


## 5. Ejecutar el pipeline JSON-first sin LLM

Este modo es conservador: genera unidades finales `N_N0{0}` y agrega heurísticas sólo como señales candidatas auditables.

In [7]:
result = run_pipeline(
    input_dir="data/transcripts/videos",
    output_dir="outputs",
    run_id="colab_no_llm_demo",
    use_llm=False,
    use_adjudicator=False,
    audit_similarity_enabled=False,
    limit=1,
)

print("run_id:", result.run_id)
print("run_dir:", result.run_dir)
print("documents:", len(result.documents))

run_id: colab_no_llm_demo
run_dir: outputs/colab_no_llm_demo
documents: 1


## 6. Leer outputs JSON/JSONL

In [13]:
run_dir = Path("outputs/colab_no_llm_demo")

manifest = json.loads((run_dir / "run_manifest.json").read_text(encoding="utf-8"))
print("manifest run_id:", manifest["run_id"])
print("taxonomy:", manifest["taxonomy_version"])

print("\nPrimeras unidades:")
for line in (run_dir / "units.jsonl").read_text(encoding="utf-8").splitlines()[:33]:
    unit = json.loads(line)
    print(unit["unit_id"], unit["final_notation"], unit["text"][:100])

print("\nAudit report preview:")
print((run_dir / "audit_report.md").read_text(encoding="utf-8")[:1000])

manifest run_id: colab_no_llm_demo
taxonomy: v1_0

Primeras unidades:
-1wOnr7JsS4_u00000 N_N0{0} Los aviones tienen algo que nosotros no tenemos.
-1wOnr7JsS4_u00001 N_N0{0} Una caja negra.
-1wOnr7JsS4_u00002 N_N0{0} Un registro mínimo, simple e indestructible que guarda lo esencial.
-1wOnr7JsS4_u00003 N_N0{0} Y quizás sea la mejor metáfora para lo que podemos hacer con nuestra propia memoria.
-1wOnr7JsS4_u00004 N_N0{0} Te propongo algo muy simple.
-1wOnr7JsS4_u00005 N_N0{0} Una vez al año, crea tu propia caja negra emocional.
-1wOnr7JsS4_u00006 N_N0{0} Te sugiero cinco preguntas, siempre las mismas.
-1wOnr7JsS4_u00007 N_N0{0} Puedes tomarlas tal cual o cambiarlas a tu medida.
-1wOnr7JsS4_u00008 N_N0{0} 1.
-1wOnr7JsS4_u00009 N_N0{0} ¿Qué aprendí este año?
-1wOnr7JsS4_u00010 N_N0{0} 2.
-1wOnr7JsS4_u00011 N_N0{0} ¿Qué logré?
-1wOnr7JsS4_u00012 N_N0{0} 3.
-1wOnr7JsS4_u00013 N_N0{0} ¿Qué me faltó?
-1wOnr7JsS4_u00014 N_N0{0} 4.
-1wOnr7JsS4_u00015 N_N0{0} ¿Qué me sorprendió?
-1wOnr7JsS4_u0001

## 7. Probar regresión golden

In [9]:
!python -m pytest tests/test_golden_regression.py -q

...                                                                      [100%]
3 passed in 0.13s


## 8. Alternativa por CLI

In [10]:
!narrative-dna run --input-dir data/transcripts/videos --output-dir outputs --run-id colab_cli_no_llm --no-llm --no-adjudicator --limit 1
!narrative-dna inspect --run-id colab_cli_no_llm

Wrote run colab_cli_no_llm to outputs/colab_cli_no_llm: 1 documents, 33 units, 3
relations, 4 chains.
Run colab_cli_no_llm (0.1.0): 1 documents, 33 units, 3 relations, 4 chains.


## 9. Opcional: usar OpenAI desde Colab Secrets

En Colab, guarda `OPENAI_API_KEY` en Secrets. Después descomenta y ejecuta esta celda para clasificar con LLM y adjudicator conservador.

In [ ]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

llm_result = run_pipeline(
    input_dir="data/transcripts/videos",
    output_dir="outputs",
    run_id="colab_llm_demo",
    use_llm=True,
    use_adjudicator=True,
    audit_similarity_enabled=True,
    limit=1,
)
print(llm_result.run_dir)


## 10. Evaluar contra synthetic gold high-confidence

Cuando tengas `outputs/<RUN_ID>/synthetic_gold_high_confidence.jsonl`, evalúa así:

In [ ]:
!narrative-dna evaluate --run-id colab_llm_demo --gold outputs/colab_llm_demo/synthetic_gold_high_confidence.jsonl